# Mapping Elevation Tiles with Different Zoom Levels and Sizes

This notebook aims to explain the process and techniques involved in mapping elevation tiles with layers of different zoom levels and sizes. 

## Overview of the Problem Space and Its Importance

The task of mapping image layers, such as satellite imagery, onto elevation grids is a foundational aspect of creating detailed and accurate 3D representations of Planet's surface. This process is crucial in a wide range of applications, from geographic information systems (GIS) to computer vision and 3D scene rendering. The core challenge in this domain arises from the need to balance the level of detail (LOD) in the imagery with performance and resource constraints, especially in dynamic environments where the viewer's perspective constantly changes.

### High LOD in Satellite Imagery

Layers, such Satellite imagery often provides a high level of detail, capturing the Body's surface with precision. When mapping these images onto elevation grids, a high LOD ensures that the resulting 3D terrain model is both accurate and visually compelling. This high-definition mapping is essential for tasks that require precise spatial data.

### The Challenge of Dynamic Perspectives

However, the necessity for high-resolution imagery presents a significant challenge in applications involving 3D scene rendering, particularly in computer vision. As the viewer's perspective changes, not all parts of the terrain require the same level of detail. Distant terrain features can be represented with lower-resolution imagery without compromising the overall quality of the scene, reducing the computational load and improving rendering performance.

### A Dynamic Approach to Image Layer Mapping

To address this issue, a dynamic approach to mapping image layers onto elevation grids is essential. This approach must adjust the LOD of imagery in real-time, based on the viewer's distance from different parts of the terrain. Such a strategy requires sophisticated logic at the GPU level, where a single elevation grid model can be manipulated alongside a stack of image layers and textures. This approach optimizes resources by reducing the number of draw calls and efficiently managing texture memory.

### Leveraging WebGL2 and WebGPU

Implementing this dynamic mapping system necessitates advanced GPU programming techniques, utilizing the capabilities of WebGL2 and the emerging WebGPU standard. These technologies provide the necessary tools for adjusting LOD dynamically, allowing for real-time optimizations that can significantly enhance the efficiency and performance of 3D rendering applications.

## Problem Inputs and Their Implications

The primary inputs to the problem of mapping image layers onto elevation grids involve:

1. **An Elevation Tile System**: This system is characterized by metrics such as `elevationTileSize`, which defines the dimensions of each tile in the system. These tiles are the basic units used to construct a detailed elevation map of the body's surface.

2. **A Consequent Grid as Mesh**: Accompanying and resulting from the elevation tile system is a mesh grid with dimensions of `ew=eh=tileSize+1`. This specific dimensioning is crucial for the construction and rendering of the elevation grid in a 3D space.

![Elevation Grid](../images/elevation_grid.png "the elevation grid reference")


### The Significance of the "+1" in Mesh Dimensions

The addition of `+1` to the mesh grid dimensions (`ew` and `eh`) is a critical aspect of this system, designed to address a common challenge in tiling systems: the seamless connection of adjacent tiles. 
In a tiled elevation grid, each tile represents a specific portion of the body's surface. However, without an overlap or a mechanism for connection, there would be visible gaps between adjacent tiles due to the discrete nature of the data and potential rounding errors in rendering. The "+1" in the dimensions of the mesh grid effectively creates an overlap between adjacent tiles, ensuring that there are no gaps in the rendered 3D terrain. This overlap allows for the vertices at the edge of one tile to seamlessly connect with the corresponding vertices of neighboring tiles, facilitating a continuous and cohesive mapping of image layers across the entire elevation grid.

In [114]:
elevationTileSize = 256
ew = elevationTileSize + 1
hw = elevationTileSize + 1
elevationLOD = 11

Depending on the desired definition and size of the map, different strategies can be employed for mapping image layers onto the elevation. Mapping can be executed using several image tiles at a higher Level of Detail (LOD), or it might involve using a lower definition, which leads to utilizing only a portion of an image.

<p align="left">
  <img src="../images/map_greater_lod.png" alt="mapping higher level of detail" style="width: auto; max-width: 48%; margin-right: 2%; vertical-align: top;">
  <img src="../images/map_lower_lod.png" alt="mapping lower level of detail" style="width: auto; max-width: 48%; vertical-align: top;">
</p>

Then we define the additional variables.<br>
<br>
⚠️ **Warning:** At this point, we assume that the tile metrics are consistent between the elevation and the mapping. This is why we can use the LOD (Level of Detail) difference to compute the varying scale and offset. However, to introduce broader support, we need access to the metrics on both tile systems and compare either geographic coverage or projected XY values - also assuming that the projections are compatible


In [115]:
import math
def fract(number):
    """
    Returns the fractional part of the given number.
    
    Parameters:
    - number (float): The input number.
    
    Returns:
    - float: The fractional part of the input number.
    """
    return number - int(number)

mapTileSize = 256
mw = mapTileSize
mh = mapTileSize
mapLOD = 13

d = mapLOD - elevationLOD 
if d > 0:
    s = 1 << d
elif d < 0:
    s = 1.0 / float(1 << abs(d))
else:
    s = 1.0
n = math.ceil(s)
print(f"d={d}, s={s}, n={n}")

d=2, s=4, n=4


By utilizing these variables defined at the CPU level, the GPU can dynamically compute the UV coordinates using the following formulas:
$$u1 = uOffset + fract(u*s)$$
$$v1 = vOffset + fract(v*s)$$

where

* $s$ is the scale
* $u,v$ are one of the UV's coordinate of the grid
* The $fract()$ function returns the fractional part of a floating point number, essentially discarding the integer part.
* $uOffset$ and $vOffset$ are the offsets added to define the origin of the mapping.

The variable `n` specifies the dimensions of a buffer, which is $[n, n]$. This buffer stores references to an image within a 2D/3D texture sampler. Consequently, the indices within this buffer are determined as follows:
$$i = floor(u1)$$
$$j= floor(v1)$$
$$0 \leq i < n, \quad i \in \mathbb{Z}_{\geq 0}$$
$$0 \leq j < n, \quad j \in \mathbb{Z}_{\geq 0}$$


